In [18]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
#read in all words
words = open('names.txt', 'r').read().splitlines()

# create lookup maps
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

32033

In [21]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X,Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append
X=torch.tensor(X)
Y=torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [22]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [23]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [47]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [48]:
C = torch.randn((27,2)).float()
C[torch.tensor([4,5,6])]

tensor([[-1.7155, -0.3402],
        [-0.8813, -2.1730],
        [ 0.4671, -0.5195]])

In [49]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [50]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
h.shape

torch.Size([32, 100])

In [51]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [10]:
logits = h @ W2 + b2

In [53]:
logits.shape

torch.Size([32, 27])

In [54]:
logits

tensor([[-5.8553e+00, -6.5057e+00, -5.4891e+00,  7.7096e+00,  8.0670e+00,
         -6.3108e+00, -1.4663e+01, -3.1961e+00, -1.1077e+01,  9.1200e-03,
          2.0902e+01,  3.8573e+00,  4.1722e+00,  3.5366e+00, -1.0334e+00,
          4.6414e+00, -1.3582e+00, -2.4287e+00, -3.4307e+00,  1.4089e+01,
          1.1837e+01, -3.0052e+00,  3.1711e-01, -1.1525e+00,  1.7492e-01,
         -7.9585e+00,  8.8844e-01],
        [-7.0931e+00,  3.0213e+00, -6.6150e+00,  1.0929e+01, -1.5278e+00,
         -1.8615e+01, -2.2999e+01, -1.5392e+00,  1.1006e+00,  4.4687e+00,
          1.7086e+01,  1.6181e+01,  4.7216e+00,  1.1492e+00, -5.7756e+00,
         -7.9563e+00,  1.0343e-01,  6.4484e+00, -1.7864e+00,  1.8612e+01,
          1.3880e+01,  9.1999e+00, -8.7938e+00, -8.3545e+00, -6.0422e-01,
         -5.1616e+00,  1.1294e+01],
        [-4.0430e+00, -8.7332e+00,  1.2858e+01,  2.1145e+01,  5.6001e+00,
         -1.5694e+01, -6.8043e+00, -4.0292e+00, -2.3431e+01,  7.1096e+00,
          1.1484e+01, -4.5112e+00,  9.79

In [55]:
counts = logits.exp()

In [56]:
prob = counts/counts.sum(1, keepdims=True)

In [57]:
prob.shape

torch.Size([32, 27])

In [58]:
prob[0].sum()

tensor(1.)

In [17]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [61]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(18.5719)

In [ ]:
### refactor

In [80]:
X.shape, Y.shape # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [81]:
g = torch.Generator().manual_seed(2147483647) 
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn((27), generator=g)
parameters = [C, W1, b1, W2, b2]

In [82]:
sum(p.nelement() for p in parameters) # num of parameters total

3481

In [83]:
emb = C[X]
h = torch.tanh(emb.view(-1,6) @ W1 +b1) # (32,100)
logits = h @ W2 + b2 #(32,27)
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdims=True)
# loss = -prob[torch.arange(32), Y].log().mean()
# loss
F.cross_entropy(logits, Y)

tensor(17.7697)